In [0]:
%pylab inline

In [0]:
import os
import re
import tempfile

import dataiku
import pandas as pd
import requests

In [0]:
# Example: load a DSS dataset as a Pandas dataframe
mydataset = dataiku.Dataset("mydataset")
mydataset_df = mydataset.get_dataframe()

In [0]:
BASE = "https://ftp.ebi.ac.uk/pub/databases/opentargets/platform/26.06/output/"
HEADERS = {"User-Agent": "Mozilla/5.0"}

def read_ot(subdir, columns=None):
    html = requests.get(BASE + subdir + "/", timeout=120, headers=HEADERS).text
    files = [f for f in re.findall(r'href="([^"]+\.parquet)"', html) if "/" not in f]
    frames = []
    for f in sorted(files):
        rr = requests.get(BASE + subdir + "/" + f, timeout=600, headers=HEADERS)
        rr.raise_for_status()
        with tempfile.NamedTemporaryFile(suffix=".parquet", delete=False) as tf:
            tf.write(rr.content)
            tmp = tf.name
        try:
            frames.append(pd.read_parquet(tmp, columns=columns))
        finally:
            os.remove(tmp)
        break
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

In [0]:
read_ot("target")